# ALF Core Quickstart: Synthetic Optimisation with alf-core

This tutorial runs a complete active learning loop using **only `alf-core`**.

We'll optimise a synthetic function over a discrete 2-D space using:
- a custom `BaseDataset` subclass backed by a numpy array
- a **bootstrap ensemble** surrogate (pure numpy)
- a **Probability of Improvement (PI)** acquisition function (scipy only)
- ALF's standard `DesignTask` to drive the loop

Both the model and acquisition function are custom implementations not found in `alf-tools`,
making this a good template for bringing your own components.

**Requirements:** `alf-core`, `numpy`, `scipy`

## Installation

Ensure the `alf_core` package is installed. You can install it via the following command:

```
pip install alf_core
```

In [ ]:
import numpy as np
from alf_core import (
    AcquisitionFunction,
    BaseModel,
    Candidate,
    DatasetSearch,
    DesignTask,
    LabelledCandidates,
    Modality,
    Optimizer,
    Oracle,
    Predictions,
    State,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from scipy.stats import norm

## 1. Define a synthetic dataset

Our search space is a discrete grid of 2-D points. The true objective is the Branin function —
a classic benchmark with two near-optimal regions. We treat each point as a `Candidate` with
a 2-element numpy array as its value.

`BaseDataset` requires two methods: `load_dataset` (returns the full labelled pool at startup)
and `query` (scores a batch of candidates — called by the oracle each round).

In [ ]:
def branin(x: np.ndarray) -> float:
    """Branin function, inverted so the global optimum is near 1.0."""
    x1, x2 = x[0] * 15 - 5, x[1] * 15  # rescale [0,1]^2 to Branin domain
    a, b, c = 1.0, 5.1 / (4 * np.pi**2), 5 / np.pi
    r, s, t = 6.0, 10.0, 1 / (8 * np.pi)
    raw = a * (x2 - b * x1**2 + c * x1 - r) ** 2 + s * (1 - t) * np.cos(x1) + s
    return float(1.0 - raw / 300)


class SyntheticDatasetConfig(BaseDatasetConfig):
    grid_size: int = 20


class SyntheticDataset(BaseDataset):
    """A discrete grid over [0,1]^2 scored by the Branin function."""

    config: SyntheticDatasetConfig

    def load_dataset(self) -> LabelledCandidates:
        rng = np.random.default_rng(self.config.seed)
        n = self.config.grid_size**2
        xs = np.array([
            (i / self.config.grid_size, j / self.config.grid_size)
            for i in range(self.config.grid_size)
            for j in range(self.config.grid_size)
        ])
        idx = rng.permutation(n)
        xs = xs[idx]
        candidates = [Candidate(data=x, modality=Modality.TABULAR) for i, x in enumerate(xs)]
        labels = np.array([branin(x) for x in xs])
        return LabelledCandidates(candidates=candidates, labels=labels)

    def query(self, candidates: list[Candidate]) -> LabelledCandidates:
        labels = np.array([branin(c.data) for c in candidates])
        return LabelledCandidates(candidates=candidates, labels=labels)

## 2. Define a bootstrap ensemble model

A bootstrap ensemble fits `n_estimators` linear models, each on a different bootstrap resample
of the training data. The mean and standard deviation across predictions give us a cheap
uncertainty estimate — no PyTorch or sklearn required.

`BaseModel` requires `featurise`, `train`, and `predict`. We also implement `sample` for
compatibility with Thompson Sampling-style acquisition functions.

In [ ]:
class BootstrapEnsemble(BaseModel):
    problem_type = ProblemType.REGRESSION
    output_dim = 1

    def __init__(self, n_estimators: int = 20, seed: int = 0) -> None:
        self.n_estimators = n_estimators
        self.rng = np.random.default_rng(seed)
        self._weights: list[np.ndarray] = []

    def featurise(self, inputs: list[Candidate]) -> np.ndarray:
        # Simple polynomial features: [1, x1, x2, x1^2, x2^2, x1*x2]
        X = np.array([c.data for c in inputs])
        return np.column_stack([np.ones(len(X)), X, X**2, X[:, 0] * X[:, 1]])

    def train(
        self,
        train_data: LabelledCandidates,
        val_data: LabelledCandidates | None = None,
        train_config=None,
    ) -> None:
        X = self.featurise(train_data.candidates)
        y = train_data.labels
        n = len(y)
        self._weights = []
        for _ in range(self.n_estimators):
            idx = self.rng.integers(0, n, size=n)
            w, *_ = np.linalg.lstsq(X[idx], y[idx], rcond=None)
            self._weights.append(w)

    def predict(self, inputs: list[Candidate]) -> Predictions:
        X = self.featurise(inputs)
        preds = np.array([X @ w for w in self._weights])  # (n_estimators, n_candidates)
        return Predictions(means=preds.mean(axis=0), variances=preds.var(axis=0))

    def sample(self, inputs: list[Candidate], n_samples: int = 1) -> np.ndarray:
        X = self.featurise(inputs)
        idx = self.rng.integers(0, self.n_estimators, size=n_samples)
        return np.array([X @ self._weights[i] for i in idx]).T

## 3. Define a Probability of Improvement acquisition function

**Probability of Improvement (PI)** scores each candidate by the probability that its true
value exceeds the current best observed value:

$$\text{PI}(x) = \Phi\left(\frac{\mu(x) - f^* - \xi}{\sigma(x)}\right)$$

where $\Phi$ is the standard normal CDF, $f^*$ is the best label seen so far, and $\xi$ is a
small exploration bonus. Unlike UCB, PI has a natural probabilistic interpretation and tends
to be more conservative (exploitation-heavy) by default.

In [ ]:
class ProbabilityOfImprovement(AcquisitionFunction):
    def __init__(self, xi: float = 0.01) -> None:
        self.xi = xi  # exploration bonus; increase to favour less-explored regions

    def __call__(self, search_candidates: list[Candidate], state: State) -> LabelledCandidates:
        predictions = state.surrogate.predict(search_candidates)
        best = state.dataset.train_dataset.labels.max()
        std = np.sqrt(predictions.variances + 1e-9)
        z = (predictions.means - best - self.xi) / std
        scores = norm.cdf(z)
        return LabelledCandidates(candidates=search_candidates, labels=scores)

## 4. Assemble and run the active learning loop

With all components defined, we wire them together using ALF's standard `DesignTask`.
The task handles the loop: train surrogate → score candidates → acquire batch → query oracle → repeat.

We update `PI._best` before each run so the acquisition function knows the current best score.

In [ ]:
config = SyntheticDatasetConfig(
    name="branin",
    modality=Modality.TABULAR,
    seed=42,
    train_ratio=0.05,  # start with 5% of the grid as labelled data
    validation_frac=0.0,
    test_ratio=0.1,
    problem_type=ProblemType.REGRESSION,
    grid_size=20,  # 400 candidates total (keep small for demo speed)
)

dataset = SyntheticDataset(config=config)
surrogate = Surrogate(model=BootstrapEnsemble(n_estimators=20, seed=0))
optimizer = Optimizer(acquisition_fn=ProbabilityOfImprovement(xi=0.01), search_fn=DatasetSearch())
oracle = Oracle(scorer=dataset)

task = DesignTask(num_acq_rounds=5, acq_batch_size=10)
dataset.setup()
state = task.setup(dataset=dataset, surrogate=surrogate)
task.run(
    state=state,
    state_loggers=[TerminalStateLogger()],
    optimizer=optimizer,
    oracle=oracle,
)

## 5. Inspect results

After the loop completes, `state` holds the full history of labelled candidates across all rounds.

In [ ]:
# state.history is a list of LabelledCandidates, one per acquisition round
all_labels = np.concatenate([lc.labels for lc in state.history])
all_candidates = [c for lc in state.history for c in lc.candidates]

best_idx = np.argmax(all_labels)
print(f"Best score found:        {all_labels[best_idx]:.4f}")
print(f"At point:                {all_candidates[best_idx].data}")
print(f"Global optimum (Branin): ~{branin(np.array([0.54, 0.15])):.4f}")

## Next steps

- Replace `BootstrapEnsemble` with a neural network from `alf-tools` (e.g. `CNNModel`)
- Try a real dataset: see the [Offline Design Tutorial](https://github.com/instadeepai/alf/blob/main/tutorials/experiments/offline_design_tutorial.ipynb)
- Add your own model or dataset: see the [How-to / Recipes](https://instadeepai.github.io/alf/how-to/index.html)